# 03_gnn_text_v2 — SOTA Traffic GNN

Upgrade over v1:
- **Gated TCN** — gated dilated convolutions for temporal
- **Diffusion GCN** — K-hop spatial aggregation
- **Adaptive adjacency** — learned road connectivity
- **Cross-modal fusion** — multi-head attention over text tokens
- **Qwen3.5-0.8B** — modern LLM text encoder (frozen + LoRA)

## Section 1: Setup & Data

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import time

from src import (
    load_train_speeds, load_train_texts, load_test_data,
    load_adjacency,
    build_windows, compute_norm_stats, normalize, denormalize,
    train_val_split, compute_mse, write_submission, validate_submission,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
s1, s2 = load_train_speeds()
t1, t2 = load_train_texts()
adj_raw = load_adjacency()

X1, T1, Y1 = build_windows(s1, t1)
X2, T2, Y2 = build_windows(s2, t2)

X_all = np.concatenate([X1, X2], axis=0).astype(np.float32)
Y_all = np.concatenate([Y1, Y2], axis=0).astype(np.float32)
T_all = np.array(T1 + T2)

mean, std = compute_norm_stats(np.concatenate([s1, s2], axis=0))
X_norm = normalize(X_all, mean, std)
Y_norm = normalize(Y_all, mean, std)

print(f"X: {X_norm.shape}, Y: {Y_norm.shape}")
print(f"X range: [{X_norm.min():.2f}, {X_norm.max():.2f}]")

In [ ]:
X_tr, _, Y_tr, X_va, _, Y_va = train_val_split(X_norm, None, Y_norm, val_frac=0.2)
T_tr = T_all[:len(X_tr)]
T_va = T_all[len(X_tr):]
print(f"Train: {X_tr.shape}, Val: {X_va.shape}")

In [ ]:
def build_adj_tensor(adj_matrix):
    adj = torch.tensor(adj_matrix, dtype=torch.float32)
    deg = adj.sum(dim=1)
    deg_inv_sqrt = torch.pow(deg, -0.5)
    deg_inv_sqrt[torch.isinf(deg_inv_sqrt)] = 0
    return deg_inv_sqrt.diag() @ adj @ deg_inv_sqrt.diag()

adj_t = build_adj_tensor(adj_raw).to(DEVICE)
print(f"Adj: {adj_t.shape}, nnz={(adj_t > 0).sum().item()}")

### Text Encoders

Two options, both pre-computed and cached:
1. **MiniLM** (baseline) — `all-MiniLM-L6-v2`, 384-dim, ~22M params
2. **Qwen3.5** (SOTA) — `Qwen/Qwen3.5-0.8B`, 1024-dim, frozen + LoRA

In [ ]:
from src import cache_save, cache_load

cached = cache_load("text_embeddings_v2")
if cached is not None:
    T_emb_tr_minilm = cached["minilm_tr"]
    T_emb_va_minilm = cached["minilm_va"]
    T_emb_tr_qwen = cached.get("qwen_tr")
    T_emb_va_qwen = cached.get("qwen_va")
    print(f"Loaded from cache:")
    print(f"  MiniLM: train={T_emb_tr_minilm.shape}, val={T_emb_va_minilm.shape}")
    if T_emb_tr_qwen is not None:
        print(f"  Qwen3.5: train={T_emb_tr_qwen.shape}, val={T_emb_va_qwen.shape}")
else:
    print("Encoding with MiniLM...")
    from sentence_transformers import SentenceTransformer
    minilm = SentenceTransformer("all-MiniLM-L6-v2")
    T_emb_tr_minilm = minilm.encode(T_tr.tolist(), batch_size=256,
                                      show_progress_bar=True, convert_to_numpy=True).astype(np.float32)
    T_emb_va_minilm = minilm.encode(T_va.tolist(), batch_size=256,
                                      show_progress_bar=True, convert_to_numpy=True).astype(np.float32)
    print(f"MiniLM: train={T_emb_tr_minilm.shape}, val={T_emb_va_minilm.shape}")

    print("Encoding with Qwen3.5-0.8B (frozen + LoRA)...")
    from transformers import AutoModel, AutoTokenizer
    from peft import get_peft_model, LoraConfig, TaskType

    QWEN_ID = "Qwen/Qwen3.5-0.8B"
    qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_ID, trust_remote_code=True)
    if qwen_tokenizer.pad_token is None:
        qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

    qwen_base = AutoModel.from_pretrained(QWEN_ID, trust_remote_code=True,
                                           torch_dtype=torch.float16, device_map=DEVICE)
    qwen_base.eval()
    for p in qwen_base.parameters():
        p.requires_grad = False

    lora_config = LoraConfig(
        task_type=TaskType.FEATURE_EXTRACTION, r=8, lora_alpha=16,
        target_modules=["q_proj", "v_proj"], lora_dropout=0.1,
    )
    qwen_model = get_peft_model(qwen_base, lora_config)

    @torch.no_grad()
    def encode_qwen(texts, tokenizer, model, device, batch_size=16):
        model.eval()
        embeddings = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            inputs = tokenizer(batch, padding=True, truncation=True, max_length=256,
                               return_tensors="pt").to(device)
            outputs = model(**inputs, output_hidden_states=True)
            hidden = outputs.hidden_states[-1]
            mask = inputs["attention_mask"].unsqueeze(-1).float()
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            embeddings.append(pooled.cpu().numpy())
        return np.concatenate(embeddings, axis=0).astype(np.float32)

    T_emb_tr_qwen = encode_qwen(T_tr.tolist(), qwen_tokenizer, qwen_model, DEVICE)
    T_emb_va_qwen = encode_qwen(T_va.tolist(), qwen_tokenizer, qwen_model, DEVICE)
    print(f"Qwen3.5: train={T_emb_tr_qwen.shape}, val={T_emb_va_qwen.shape}")

    cache_save("text_embeddings_v2",
               minilm_tr=T_emb_tr_minilm, minilm_va=T_emb_va_minilm,
               qwen_tr=T_emb_tr_qwen, qwen_va=T_emb_va_qwen)


### DataLoader Helpers

In [ ]:
class TrafficDS(Dataset):
    def __init__(self, X, T_emb, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.T = torch.tensor(T_emb, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.T[i], self.Y[i]


def make_loaders(T_emb_tr, T_emb_va, batch_size=32):
    train_ds = TrafficDS(X_tr, T_emb_tr, Y_tr)
    val_ds = TrafficDS(X_va, T_emb_va, Y_va)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, pin_memory=True)
    return train_loader, val_loader


print("Section 1 complete.")

## Section 2: Model Components

All defined inline as `nn.Module` classes.

In [ ]:
class GatedTCN(nn.Module):
    """Stack of gated dilated 1D convolutions with skip connections."""
    def __init__(self, in_steps=15, residual_channels=64, skip_channels=64,
                 num_layers=4, kernel_size=2, dropout=0.1):
        super().__init__()
        self.num_layers = num_layers
        self.skip_convs = nn.ModuleList()
        self.filter_convs = nn.ModuleList()
        self.gate_convs = nn.ModuleList()

        in_dim = residual_channels
        for i in range(num_layers):
            dilation = 2 ** i
            padding = (kernel_size - 1) * dilation
            self.filter_convs.append(
                nn.Conv1d(in_dim, residual_channels, kernel_size,
                          dilation=dilation, padding=padding))
            self.gate_convs.append(
                nn.Conv1d(in_dim, residual_channels, kernel_size,
                          dilation=dilation, padding=padding))
            self.skip_convs.append(nn.Conv1d(residual_channels, skip_channels, 1))

        self.start_conv = nn.Conv1d(1, residual_channels, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.start_conv(x)
        skips = []

        for i in range(self.num_layers):
            residual = x
            f = torch.tanh(self.filter_convs[i](x))
            g = torch.sigmoid(self.gate_convs[i](x))
            x = f * g
            x = x[:, :, :residual.size(2)]
            x = x + residual[:, :, :x.size(2)]
            skip = self.skip_convs[i](x)  # (B*N, skip_c, T)
            skips.append(skip[:, :, -1:])  # (B*N, skip_c, 1)
            x = self.dropout(x)

        return x[:, :, -1], skips  # (B*N, C), list of (B*N, skip_c, 1)


In [ ]:
class DiffusionGCN(nn.Module):
    """K-hop diffusion graph convolution."""
    def __init__(self, in_channels, out_channels, num_hops=2, dropout=0.1):
        super().__init__()
        self.num_hops = num_hops
        self.weights = nn.Parameter(
            torch.empty(num_hops, in_channels, out_channels))
        nn.init.xavier_uniform_(self.weights)
        self.norm = nn.LayerNorm(out_channels)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, adj):
        out = 0
        h = x
        for k in range(self.num_hops):
            h_k = h if k == 0 else adj @ h
            out_k = torch.einsum("bnc,co->bno", h_k, self.weights[k])
            out += out_k
            h = adj @ h
        return self.drop(F.relu(self.norm(out)))


In [ ]:
class AdaptiveAdj(nn.Module):
    """Learned adjacency matrix, combined with static road network."""
    def __init__(self, num_nodes=1260, embed_dim=10):
        super().__init__()
        self.src_emb = nn.Embedding(num_nodes, embed_dim)
        self.dst_emb = nn.Embedding(num_nodes, embed_dim)

    def forward(self, static_adj):
        N = static_adj.size(0)
        idx = torch.arange(N, device=static_adj.device)
        s = self.src_emb(idx)
        d = self.dst_emb(idx)
        a = F.softmax(F.relu(s @ d.T), dim=-1)
        deg = a.sum(dim=1)
        deg_inv_sqrt = torch.pow(deg, -0.5)
        deg_inv_sqrt[torch.isinf(deg_inv_sqrt)] = 0
        a_norm = deg_inv_sqrt.diag() @ a @ deg_inv_sqrt.diag()
        return static_adj + a_norm


In [ ]:
class CrossModalFusion(nn.Module):
    """Multi-head cross-attention: node features query text tokens."""
    def __init__(self, d_model, d_text, n_heads=4, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = self.d_head ** -0.5

        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_text, d_model)
        self.W_V = nn.Linear(d_text, d_model)
        self.W_O = nn.Linear(d_model, d_model)

        self.gate = nn.Linear(d_model * 2, 1)
        self.drop = nn.Dropout(dropout)

    def forward(self, node_feat, text_feat):
        B, N, _ = node_feat.shape
        S = text_feat.size(1)

        Q = self.W_Q(node_feat).reshape(B, N, self.n_heads, self.d_head)
        Q = Q.permute(0, 2, 1, 3)

        K = self.W_K(text_feat).reshape(B, S, self.n_heads, self.d_head)
        K = K.permute(0, 2, 1, 3)

        V = self.W_V(text_feat).reshape(B, S, self.n_heads, self.d_head)
        V = V.permute(0, 2, 1, 3)

        attn = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        attn = F.softmax(attn, dim=-1)
        attn = self.drop(attn)

        out = torch.matmul(attn, V)
        out = out.permute(0, 2, 1, 3).reshape(B, N, -1)
        out = self.W_O(out)

        combined = torch.cat([node_feat, out], dim=-1)
        g = torch.sigmoid(self.gate(combined))
        return g * node_feat + (1 - g) * out


In [ ]:
# Smoke test: instantiate and forward-pass each component
print("GatedTCN:")
tcn = GatedTCN(in_steps=15, residual_channels=64, skip_channels=64, num_layers=4, dropout=0.1)
x = torch.randn(10, 1, 15)  # (B*N, 1, T)
out_tcn, skips_tcn = tcn(x)
print(f"  Input: {x.shape} -> Output: {out_tcn.shape}, Skips: {len(skips_tcn)}x{skips_tcn[0].shape}")

print("DiffusionGCN:")
dgcn = DiffusionGCN(64, 64, num_hops=2, dropout=0.1)
adj_dummy = torch.eye(1260) * 0.5
x_g = torch.randn(2, 1260, 64)
out_g = dgcn(x_g, adj_dummy)
print(f"  Input: {x_g.shape} -> Output: {out_g.shape}")

print("AdaptiveAdj:")
aadj = AdaptiveAdj(1260, embed_dim=10)
adj_out = aadj(adj_dummy)
print(f"  Static: {adj_dummy.shape} -> Mixed: {adj_out.shape}")

print("CrossModalFusion:")
cmf = CrossModalFusion(d_model=64, d_text=384, n_heads=4, dropout=0.1)
node_f = torch.randn(2, 1260, 64)
text_f = torch.randn(2, 16, 384)
out_f = cmf(node_f, text_f)
print(f"  Node: {node_f.shape}, Text: {text_f.shape} -> Output: {out_f.shape}")

print("\nAll components OK.")


## Section 3: SOTATrafficGNN Assembly

Combines all four components into a single model.

```
Speed (B,15,1260)                   Text tokens (B, S, d_text)
    │                                       │
    ▼                                       │
reshape (B*N, 1, 15)                        │
    │                                       │
    ▼                                       │
GatedTCN × L_t layers                       │
    │   └── skipped                          │
    ▼                                       │
reshape (B, N, d_model)                     │
    │                                       │
    ├──── CrossModalFusion ◄─────────────────┘
    │                                       │
    ▼                                       │
adj = AdaptiveAdj(static_adj)               │
    │                                       │
    ▼                                       │
DiffusionGCN × L_g layers                   │
    │   └── skips collected                  │
    ▼                                       │
concat(all skips) → (B,N, d * (L_t+L_g))    │
    │                                       │
    ▼                                       │
Decoder MLP → (B,N,3) → (B,3,N)
```

In [ ]:
class SOTATrafficGNN(nn.Module):
    def __init__(self, num_nodes=1260, in_steps=15, out_horizons=3,
                 d_model=64, num_tcn_layers=4, num_gcn_layers=4,
                 num_hops=2, text_dim=384, n_heads=4,
                 dropout=0.2, use_text=True, use_adaptive_adj=True,
                 fusion="cross"):
        super().__init__()
        self.use_text = use_text
        self.use_adaptive_adj = use_adaptive_adj
        self.fusion = fusion
        self.num_nodes = num_nodes
        self.num_tcn_layers = num_tcn_layers
        self.num_gcn_layers = num_gcn_layers

        # --- Temporal ---
        if num_tcn_layers > 0:
            self.tcn = GatedTCN(
                in_steps=in_steps,
                residual_channels=d_model,
                skip_channels=d_model,
                num_layers=num_tcn_layers,
                kernel_size=2,
                dropout=dropout,
            )
        else:
            # Fallback: simple Conv1D + pool (baseline)
            self.tcn = None
            self.time_enc = nn.Sequential(
                nn.Conv1d(1, d_model, kernel_size=5, padding=2),
                nn.ReLU(),
                nn.AdaptiveAvgPool1d(1),
            )
            num_tcn_layers = 1  # 1 skip from simple encoder

        # --- Text ---
        if use_text:
            if fusion == "cross":
                self.cross_modal = CrossModalFusion(d_model, text_dim, n_heads, dropout)
            else:
                self.text_proj = nn.Sequential(
                    nn.Linear(text_dim, d_model),
                    nn.ReLU(),
                )
                self.text_fuse = nn.Linear(d_model * 2, d_model)

        # --- Spatial ---
        if use_adaptive_adj:
            self.adaptive_adj = AdaptiveAdj(num_nodes, embed_dim=10)

        self.gcn_layers = nn.ModuleList([
            DiffusionGCN(d_model, d_model, num_hops, dropout)
            for _ in range(num_gcn_layers)
        ])

        # --- Decoder ---
        total_skips = num_tcn_layers + num_gcn_layers
        decoder_in = d_model * total_skips
        self.decoder = nn.Sequential(
            nn.Linear(decoder_in, d_model * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, out_horizons),
        )

    def forward(self, x, text_feat, adj):
        # x: (B, T, N), text_feat: (B, S, d_text) or (B, d_text)
        # adj: (N, N) static normalized adjacency
        B, T, N = x.shape

        # --- Temporal ---
        h = x.permute(0, 2, 1).reshape(B * N, 1, T)  # (B*N, 1, T)
        if self.tcn is not None:
            h, tcn_skip_list = self.tcn(h)
            h = h.reshape(B, N, -1)  # (B, N, d)
            tcn_skips = [s.reshape(B, N, -1) for s in tcn_skip_list]
        else:
            h = self.time_enc(h).squeeze(-1)  # (B*N, d)
            h = h.reshape(B, N, -1)  # (B, N, d)
            tcn_skips = [h]  # single skip from simple encoder

        # --- Text fusion ---
        if self.use_text:
            if self.fusion == "cross":
                if text_feat.dim() == 2:
                    text_feat = text_feat.unsqueeze(1)
                h = self.cross_modal(h, text_feat)
            else:
                if text_feat.dim() == 3:
                    text_feat = text_feat.mean(dim=1)  # pool tokens
                t = self.text_proj(text_feat)  # (B, d)
                t = t.unsqueeze(1).expand(-1, N, -1)  # (B, N, d)
                h = F.relu(self.text_fuse(torch.cat([h, t], dim=-1)))

        # --- Adaptive adjacency ---
        if self.use_adaptive_adj:
            adj = self.adaptive_adj(adj)

        # --- Spatial (GCN) ---
        gcn_skips = []
        for gcn in self.gcn_layers:
            h = gcn(h, adj)
            gcn_skips.append(h)

        # --- Decode ---
        all_skips = tcn_skips + gcn_skips
        h = torch.cat(all_skips, dim=-1)
        out = self.decoder(h)
        return out.permute(0, 2, 1)


In [ ]:
print("Testing SOTATrafficGNN with MiniLM text (384-dim):")
model = SOTATrafficGNN(
    num_nodes=1260, in_steps=15, out_horizons=3,
    d_model=64, num_tcn_layers=4, num_gcn_layers=4,
    num_hops=2, text_dim=384, n_heads=4, dropout=0.2,
    use_text=True, use_adaptive_adj=True,
)
n_params = sum(p.numel() for p in model.parameters())
print(f"  Parameters: {n_params:,}")

x_dummy = torch.randn(2, 15, 1260)
t_dummy = torch.randn(2, 384)  # sentence embedding (no token dim)
adj_dummy = torch.eye(1260) * 0.5

with torch.no_grad():
    out = model(x_dummy, t_dummy, adj_dummy)
print(f"  Input: {x_dummy.shape}, Text: {t_dummy.shape}")
print(f"  Output: {out.shape} (expected: [2, 3, 1260])")

print("\nTesting with Qwen3.5 text (token-level, 1024-dim):")
model_qw = SOTATrafficGNN(
    num_nodes=1260, in_steps=15, out_horizons=3,
    d_model=64, num_tcn_layers=4, num_gcn_layers=4,
    num_hops=2, text_dim=1024, n_heads=4, dropout=0.2,
    use_text=True, use_adaptive_adj=True,
)
t_tokens = torch.randn(2, 32, 1024)  # token-level embeddings
with torch.no_grad():
    out_qw = model_qw(x_dummy, t_tokens, adj_dummy)
print(f"  Text: {t_tokens.shape}")
print(f"  Output: {out_qw.shape} (expected: [2, 3, 1260])")

print("\nTesting speed-only (no text):")
model_so = SOTATrafficGNN(
    use_text=False, use_adaptive_adj=True,
)
with torch.no_grad():
    out_so = model_so(x_dummy, t_dummy, adj_dummy)
n_so = sum(p.numel() for p in model_so.parameters())
print(f"  Parameters: {n_so:,}")
print(f"  Output: {out_so.shape}")

print("\nAll SOTATrafficGNN variants OK.")


## Section 4: Ablation Table

Train each config and compare validation MSE.

| # | Name | Temporal | Spatial | Adjacency | Fusion | Text |
|---|---|---|---|---|---|---|
| A | Baseline (v1) | Conv1D+pool | 1-hop GCN | Static | Concat | MiniLM |
| B | + Gated TCN | GatedTCN | 1-hop GCN | Static | Concat | MiniLM |
| C | + Diffusion | GatedTCN | K-hop (K=2) | Static | Concat | MiniLM |
| D | + Adaptive | GatedTCN | K-hop (K=2) | Learned | Concat | MiniLM |
| E | + Cross-modal | GatedTCN | K-hop (K=2) | Learned | Cross-attn | MiniLM |
| F | + Qwen3.5 | GatedTCN | K-hop (K=2) | Learned | Cross-attn | Qwen3.5 |

In [ ]:
def train_one_config(model, train_loader, val_loader, adj, device,
                     epochs=50, lr=1e-3, patience=10, verbose=False):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    best_val = float("inf")
    best_state = None
    patience_left = patience

    for epoch in range(1, epochs + 1):
        model.train()
        for Xb, Tb, Yb in train_loader:
            Xb, Tb, Yb = Xb.to(device), Tb.to(device), Yb.to(device)
            # Handle 2D vs 3D text embeddings transparently
            pred = model(Xb, Tb, adj)
            loss = criterion(pred, Yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for Xb, Tb, Yb in val_loader:
                Xb, Tb, Yb = Xb.to(device), Tb.to(device), Yb.to(device)
                pred = model(Xb, Tb, adj)
                val_loss += criterion(pred, Yb).item() * Xb.size(0)
        val_loss /= len(val_loader.dataset)

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_left = patience
        else:
            patience_left -= 1
            if patience_left == 0:
                break

    model.load_state_dict(best_state)
    n_params = sum(p.numel() for p in model.parameters())
    return best_val, n_params, epoch

print("Training helper ready.")

In [ ]:
@torch.no_grad()
def eval_mse(model, loader, adj, device):
    model.eval()
    preds, targets = [], []
    for Xb, Tb, Yb in loader:
        Xb, Tb, Yb = Xb.to(device), Tb.to(device), Yb.to(device)
        pb = model(Xb, Tb, adj)
        preds.append(pb.cpu().numpy())
        targets.append(Yb.cpu().numpy())
    yp_norm = np.concatenate(preds, axis=0)
    yt_norm = np.concatenate(targets, axis=0)
    yp = denormalize(yp_norm, mean, std)
    yt = denormalize(yt_norm, mean, std)
    return compute_mse(yp, yt)

print("Evaluation helper ready.")

In [ ]:
results = {}

# --- A: Baseline (v1-style: simple Conv1D + static GCN + concat fusion + MiniLM) ---
print("A: Baseline (v1-style, MiniLM concat)...")
model_a = SOTATrafficGNN(
    d_model=64, num_tcn_layers=0, num_gcn_layers=4,
    num_hops=1, text_dim=384, n_heads=4, dropout=0.2,
    use_text=True, fusion="concat", use_adaptive_adj=False,
).to(DEVICE)
tl_a, vl_a = make_loaders(T_emb_tr_minilm, T_emb_va_minilm, batch_size=32)
mse_a, n_a, ep_a = train_one_config(model_a, tl_a, vl_a, adj_t, DEVICE,
                                      epochs=50, patience=10)
mse_a_real = eval_mse(model_a, vl_a, adj_t, DEVICE)
results["A: Baseline"] = mse_a_real
print(f"  -> Val MSE: {mse_a_real:.4f} ({n_a:,} params, {ep_a} epochs)
")

# --- B: + Gated TCN ---
print("B: + GatedTCN (MiniLM concat, 1-hop, static adj)...")
model_b = SOTATrafficGNN(
    d_model=64, num_tcn_layers=4, num_gcn_layers=4,
    num_hops=1, text_dim=384, n_heads=4, dropout=0.2,
    use_text=True, fusion="concat", use_adaptive_adj=False,
).to(DEVICE)
tl_b, vl_b = make_loaders(T_emb_tr_minilm, T_emb_va_minilm, batch_size=32)
mse_b, n_b, ep_b = train_one_config(model_b, tl_b, vl_b, adj_t, DEVICE,
                                      epochs=50, patience=10)
mse_b_real = eval_mse(model_b, vl_b, adj_t, DEVICE)
results["B: + GatedTCN"] = mse_b_real
print(f"  -> Val MSE: {mse_b_real:.4f} ({n_b:,} params, {ep_b} epochs)
")

# --- C: + Diffusion GCN (K=2) ---
print("C: + Diffusion K=2 (MiniLM concat, static adj)...")
model_c = SOTATrafficGNN(
    d_model=64, num_tcn_layers=4, num_gcn_layers=4,
    num_hops=2, text_dim=384, n_heads=4, dropout=0.2,
    use_text=True, fusion="concat", use_adaptive_adj=False,
).to(DEVICE)
tl_c, vl_c = make_loaders(T_emb_tr_minilm, T_emb_va_minilm, batch_size=32)
mse_c, n_c, ep_c = train_one_config(model_c, tl_c, vl_c, adj_t, DEVICE,
                                      epochs=50, patience=10)
mse_c_real = eval_mse(model_c, vl_c, adj_t, DEVICE)
results["C: + Diffusion"] = mse_c_real
print(f"  -> Val MSE: {mse_c_real:.4f} ({n_c:,} params, {ep_c} epochs)
")

# --- D: + Adaptive Adjacency ---
print("D: + AdaptiveAdj (MiniLM concat, K=2)...")
model_d = SOTATrafficGNN(
    d_model=64, num_tcn_layers=4, num_gcn_layers=4,
    num_hops=2, text_dim=384, n_heads=4, dropout=0.2,
    use_text=True, fusion="concat", use_adaptive_adj=True,
).to(DEVICE)
tl_d, vl_d = make_loaders(T_emb_tr_minilm, T_emb_va_minilm, batch_size=32)
mse_d, n_d, ep_d = train_one_config(model_d, tl_d, vl_d, adj_t, DEVICE,
                                      epochs=50, patience=10)
mse_d_real = eval_mse(model_d, vl_d, adj_t, DEVICE)
results["D: + AdaptiveAdj"] = mse_d_real
print(f"  -> Val MSE: {mse_d_real:.4f} ({n_d:,} params, {ep_d} epochs)
")

# --- E: + Cross-modal attention ---
print("E: + CrossModal (MiniLM cross-attn, K=2, adaptive)...")
model_e = SOTATrafficGNN(
    d_model=64, num_tcn_layers=4, num_gcn_layers=4,
    num_hops=2, text_dim=384, n_heads=4, dropout=0.2,
    use_text=True, fusion="cross", use_adaptive_adj=True,
).to(DEVICE)
tl_e, vl_e = make_loaders(T_emb_tr_minilm, T_emb_va_minilm, batch_size=32)
mse_e, n_e, ep_e = train_one_config(model_e, tl_e, vl_e, adj_t, DEVICE,
                                      epochs=50, patience=10)
mse_e_real = eval_mse(model_e, vl_e, adj_t, DEVICE)
results["E: + CrossModal"] = mse_e_real
print(f"  -> Val MSE: {mse_e_real:.4f} ({n_e:,} params, {ep_e} epochs)
")

# --- F: + Qwen3.5 encoder ---
print("F: + Qwen3.5 (cross-attn, K=2, adaptive)...")
model_f = SOTATrafficGNN(
    d_model=64, num_tcn_layers=4, num_gcn_layers=4,
    num_hops=2, text_dim=1024, n_heads=4, dropout=0.2,
    use_text=True, fusion="cross", use_adaptive_adj=True,
).to(DEVICE)
tl_f, vl_f = make_loaders(T_emb_tr_qwen, T_emb_va_qwen, batch_size=16)
mse_f, n_f, ep_f = train_one_config(model_f, tl_f, vl_f, adj_t, DEVICE,
                                      epochs=50, patience=10)
mse_f_real = eval_mse(model_f, vl_f, adj_t, DEVICE)
results["F: + Qwen3.5"] = mse_f_real
print(f"  -> Val MSE: {mse_f_real:.4f} ({n_f:,} params, {ep_f} epochs)
")

print("=" * 60)
print(f"{"Config":<30} {"Val MSE":>10} {"Params":>12} {"Epochs":>8}")
print("-" * 64)
for name, mse in results.items():
    print(f"{name:<30} {mse:>10.4f}")

best_name = min(results, key=results.get)
print(f"
Best: {best_name} ({results[best_name]:.4f})")